# 03 · Baselines and drift: NYC Yellow Taxi

Phase 3, Step 3.7. Shows the latest run of the dataset-level checks on the real months and
confirms the calibration: the January → February 2023 schema change is found, 2019-01 is above
the cold-start volume limits, and the clean 2023 months raise no volume or drift alarms (only the
September NULL increase is expected, as **low**).

**Before running:**
```
python src/ingestion/ingest_data.py
python src/quality/run_checks.py
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config
from quality import run_checks
from quality.dataset_checks import check_settings

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)

## Settings

In [ ]:
config = load_datasets_config()
dataset = config.dataset("yellow_taxi")
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", config.paths["duckdb"]))
settings = check_settings()

con = db.connect(DATABASE, read_only=True)
row = con.execute(
    "SELECT max(run_id) FROM dataset_check_results WHERE dataset = ?", [dataset.name]
).fetchone()
assert row and row[0], "no check run found: run python src/quality/run_checks.py first"
run_id = row[0]
print("latest check run:", run_id)
results = con.execute(
    'SELECT * FROM dataset_check_results WHERE run_id = ? ORDER BY period, "check", column_name',
    [run_id],
).df()

## 1 · Overview: one line per month

In [ ]:
run_checks.overview(con, run_id)

In [ ]:
findings = run_checks.findings(con, run_id)
print(f"{len(findings)} finding(s)")
findings

## 2 · Volume against the baseline

Rows per day per month, with the baseline median and the ±30% band where a trusted baseline exists.

In [ ]:
volume = con.execute(
    """
    SELECT m.period, m.value AS rows_per_day, b.value AS baseline, b.trusted
    FROM dataset_metrics m
    JOIN (SELECT period, sha256 FROM dataset_check_results WHERE run_id = ? GROUP BY ALL) r
      USING (period, sha256)
    LEFT JOIN baselines b
      ON b.run_id = ? AND b.period = m.period AND b.metric = 'rows_per_day'
    WHERE m.metric = 'rows_per_day' AND m.dataset = ?
    ORDER BY m.period
    """,
    [run_id, run_id, dataset.name],
).df()
volume["change_pct"] = (100 * (volume["rows_per_day"] / volume["baseline"] - 1)).round(1)
volume

In [ ]:
fig = go.Figure()
fig.add_bar(x=volume["period"], y=volume["rows_per_day"], name="rows per day")
trusted = volume.dropna(subset=["baseline"])
tolerance = settings.volume_tolerance
fig.add_scatter(x=trusted["period"], y=trusted["baseline"], mode="lines+markers", name="baseline")
fig.add_scatter(x=trusted["period"], y=trusted["baseline"] * (1 + tolerance), mode="lines",
                line={"dash": "dot"}, name=f"+{tolerance:.0%}")
fig.add_scatter(x=trusted["period"], y=trusted["baseline"] * (1 - tolerance), mode="lines",
                line={"dash": "dot"}, name=f"-{tolerance:.0%}")
fig.update_layout(title="Rows per day against the baseline")
fig.show()

## 3 · NULL rates

Columns with any NULLs. The September increase in the trip-metadata group is the one change
expected to be reported (low).

In [ ]:
nulls = results[results["check"] == "null_rate"].copy()
nulls["observed_pct"] = nulls["observed"].str.rstrip("%").astype(float)
with_nulls = nulls.groupby("column_name")["observed_pct"].max()
nulls = nulls[nulls["column_name"].isin(with_nulls[with_nulls > 0].index)]
nulls.pivot(index="period", columns="column_name", values="observed_pct")

In [ ]:
nulls[nulls["status"] == "fail"][["period", "column_name", "severity", "observed", "expected", "detail"]]

## 4 · Drift (PSI) per month

Medium from PSI 0.1, high from 0.2 (`configs/quality_rules.yml`). Months without a trusted
baseline are skipped. This table confirms open item O3 on the real data.

In [ ]:
drift = results[(results["check"] == "drift") & (results["status"] != "skipped")].copy()
drift["psi"] = drift["observed"].str.removeprefix("PSI ").astype(float)
psi_table = drift.pivot(index="period", columns="column_name", values="psi")
psi_table

In [ ]:
if not drift.empty:
    fig = px.line(drift, x="period", y="psi", color="column_name", markers=True,
                  title="PSI against the baseline")
    fig.add_hline(y=settings.psi_medium, line_dash="dot", annotation_text="medium")
    fig.add_hline(y=settings.psi_high, line_dash="dot", annotation_text="high")
    fig.show()

## 5 · Schema per month

In [ ]:
schema = results[results["check"] == "schema"][["period", "status", "severity", "detail", "baseline"]]
schema

## 6 · Calibration verdict

Compares the findings with what Phase 3 expects on the full data. Copy the printed text back.

In [ ]:
expected = {
    ("2023-02", "schema", ""),               # airport_fee renamed, five type changes
    ("2019-01", "volume", ""),               # above the cold-start limits
    ("2023-09", "null_rate", "passenger_count"),
    ("2023-09", "null_rate", "RatecodeID"),  # the trip-metadata group moves together
    ("2023-09", "null_rate", "store_and_fwd_flag"),
    ("2023-09", "null_rate", "congestion_surcharge"),
    ("2023-09", "null_rate", "airport_fee"),
}
found = {(f.period, f.check, f.column_name) for f in findings.itertuples()}
unexpected = sorted(found - expected)
missing = sorted(e for e in expected - found if e[0] in set(results["period"]))
verdict = {
    "run": run_id,
    "months": sorted(results["period"].unique()),
    "findings": len(findings),
    "expected_but_missing": missing,
    "unexpected": [
        (p, c, col, findings[(findings.period == p) & (findings.check == c)
                             & (findings.column_name == col)].iloc[0][["severity", "observed"]].tolist())
        for p, c, col in unexpected
    ],
    "max_psi": psi_table.max().round(3).to_dict() if not drift.empty else {},
}
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
con.close()